# 📰 Step 5a — GDELT News Extraction (Fixed)

**Fix applied:** GDELT queries now use simple keyword phrases without OR operators and double quotes,
which caused empty JSON responses. Each topic uses a single clean search phrase.

**Output:**
- `step5a_raw_news_all_topics.csv` — all articles combined
- `gdelt_raw_by_topic/` — one CSV per topic

---
Run all cells in order: **Runtime → Run All**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install requests pandas -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Configuration
#
# KEY FIX: GDELT queries must be simple keyword phrases.
# Do NOT use:
#   ❌ Double quotes around phrases  → "Nifty 50"  causes empty response
#   ❌ OR keyword                    → Nifty OR Sensex  causes empty response
# DO use:
#   ✅ Simple space-separated words  → Nifty 50 India
#   ✅ Multiple topics run separately → one topic per query
# ─────────────────────────────────────────────────────────────────────────
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import os
from google.colab import files

# ── DATE RANGE — change to match your OHLC data ───────────────────────────
START_DATE = datetime(2023, 1, 1)    # 01-Jan-2023
END_DATE   = datetime(2026, 3, 30)   # 30-Mar-2026

# ── GDELT API ──────────────────────────────────────────────────────────────
GDELT_BASE   = 'https://api.gdeltproject.org/api/v2/doc/doc'
MAX_RECORDS  = 250
CHUNK_DAYS   = 90     # 90-day chunks — only 6 requests per topic
SLEEP_CHUNK  = 8.0    # 8s between chunks — well above 5s limit
SLEEP_TOPIC  = 10.0   # 10s between topics
SLEEP_429    = 60.0   # 60s wait on rate limit

OUTPUT_DIR = 'gdelt_raw_by_topic'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── TOPICS — simple keyword phrases only (no OR, no double quotes) ─────────
TOPICS = {

    # Top 10 high-signal topics for Indian equity market prediction
    # Covers ~80% of predictive signal based on financial research

    'NIFTY_50':            'Nifty India stock market',
    'RBI_REPO_RATE':       'Reserve Bank India repo rate policy',
    'USD_INR':             'dollar rupee exchange rate India currency',
    'CRUDE_OIL':           'Brent crude oil price India energy',
    'US_FED':              'Federal Reserve interest rate policy meeting',
    'FII_DII_FLOWS':       'foreign institutional investor India inflows',
    'GLOBAL_MARKETS':      'global stock markets Wall Street rally',
    'INDIA_CPI_INFLATION': 'India inflation consumer price index',
    'RELIANCE_INDUSTRIES': 'Reliance Industries stock market India',
    'HDFC_BANK':           'HDFC Bank India quarterly results',

}

print('=' * 60)
print('  STEP 5a — GDELT NEWS EXTRACTION (FIXED)')
print('=' * 60)
print(f'  Start     : {START_DATE.date()}')
print(f'  End       : {END_DATE.date()}')
print(f'  Topics    : {len(TOPICS)}')
print(f'  Chunk     : {CHUNK_DAYS} days per request')
days = (END_DATE - START_DATE).days
chunks = (days // CHUNK_DAYS) + 1
est   = (len(TOPICS) * chunks * (SLEEP_CHUNK + 1)) / 60
print(f'  Rate limit : 1 request per 5s (GDELT policy)')
print(f'  Est. time : ~{est:.0f}–{est*1.5:.0f} minutes')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Test GDELT Connectivity
# ─────────────────────────────────────────────────────────────────────────
print('Testing GDELT connectivity...')
print('  Waiting 8s before first request...')
time.sleep(8)

test_params = {
    'query':         'Nifty India stock market sourcelang:english',
    'mode':          'artlist',
    'maxrecords':    5,
    'startdatetime': '20250101000000',
    'enddatetime':   '20250331000000',
    'format':        'json',
    'sort':          'DateDesc',
}

def try_gdelt(params, wait_label=''):
    if wait_label:
        print(f'  {wait_label}')
    resp = requests.get(GDELT_BASE, params=params, timeout=30)
    return resp

try:
    resp = try_gdelt(test_params)
    print(f'  HTTP Status   : {resp.status_code}')
    print(f'  Response size : {len(resp.text)} chars')

    # Retry once on 429
    if resp.status_code == 429:
        resp = try_gdelt(test_params, 'Rate limited — waiting 60s then retrying...')
        time.sleep(60)
        resp = requests.get(GDELT_BASE, params=test_params, timeout=30)
        print(f'  Retry status  : {resp.status_code}')

    if resp.status_code == 200 and len(resp.text.strip()) > 10:
        try:
            data     = resp.json()
            articles = data.get('articles', [])
            print(f'  Articles found: {len(articles)}')
            if articles:
                print(f'  Sample title  : {articles[0].get("title", "")[:70]}')
            print()
            print('  ✅ GDELT is working — run Cell 4 now')
            print('  ℹ️  Cell 4 waits 8s between each request automatically')
        except Exception as je:
            print(f'  ⚠️  JSON error: {je}')
            print(f'  Raw: {resp.text[:150]}')
    elif resp.status_code == 429:
        print(f'  Response: {resp.text[:150]}')
        print()
        print('  ❌ Still rate limited — Colab IP is temporarily banned by GDELT')
        print('  Fix: Runtime → Disconnect and delete runtime → Reconnect')
        print('  This gives you a fresh IP address')
    else:
        print(f'  Response: {resp.text[:150]}')
        print()
        print('  ❌ Unexpected response')
        print('  Try: Runtime → Factory reset runtime → then run again')

except Exception as e:
    print(f'  ❌ Connection error: {e}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Fetch Function and Run All 30 Topics
# ─────────────────────────────────────────────────────────────────────────

def fetch_gdelt(topic_name, query, start_dt, end_dt):
    all_articles  = []
    current_start = start_dt

    while current_start < end_dt:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS), end_dt)
        start_str   = current_start.strftime('%Y%m%d%H%M%S')
        end_str     = current_end.strftime('%Y%m%d%H%M%S')

        params = {
            'query':         f'{query} sourcelang:english',
            'mode':          'artlist',
            'maxrecords':    MAX_RECORDS,
            'startdatetime': start_str,
            'enddatetime':   end_str,
            'format':        'json',
            'sort':          'DateDesc',
        }

        retry = 0
        while retry < 3:
            try:
                resp = requests.get(GDELT_BASE, params=params, timeout=30)

                # Rate limit — wait longer before retry
                if resp.status_code == 429:
                    print(f'    ⚠️  Rate limit (429) — waiting {SLEEP_429:.0f}s before retry {retry+1}/3')
                    time.sleep(SLEEP_429)
                    retry += 1
                    continue

                # Check for empty response body
                if len(resp.text.strip()) == 0:
                    print(f'    ⚠️  Empty response — waiting 6s then retry {retry+1}/3')
                    time.sleep(6)
                    retry += 1
                    continue

                if resp.status_code != 200:
                    print(f'    ⚠️  HTTP {resp.status_code} — skipping')
                    break

                # Parse JSON safely
                try:
                    data     = resp.json()
                except Exception:
                    print(f'    ⚠️  JSON parse error — response: {resp.text[:100]} — retry {retry+1}/3')
                    time.sleep(6)
                    retry += 1
                    continue

                articles = data.get('articles', [])
                for a in articles:
                    all_articles.append({
                        'topic':        topic_name,
                        'published_at': a.get('seendate', ''),
                        'source':       a.get('domain', ''),
                        'title':        a.get('title', ''),
                        'url':          a.get('url', ''),
                        'language':     a.get('language', ''),
                        'tone':         a.get('tone', None),
                        'country':      a.get('sourcecountry', ''),
                    })

                label = f"{current_start.strftime('%d-%b')}–{current_end.strftime('%d-%b')}"
                print(f'    {label}: {len(articles):>3} articles  (total: {len(all_articles)})')
                break  # success — move to next chunk

            except requests.exceptions.Timeout:
                print(f'    ⚠️  Timeout — retry {retry+1}/3')
                retry += 1
                time.sleep(5)
            except Exception as e:
                print(f'    ⚠️  Error: {e} — retry {retry+1}/3')
                retry += 1
                time.sleep(5)

        current_start = current_end
        time.sleep(SLEEP_CHUNK)

    return all_articles


# ── Run all topics ─────────────────────────────────────────────────────────
all_articles  = []
topic_summary = {}
failed_topics = []

print('=' * 60)
print('  FETCHING ALL 30 TOPICS FROM GDELT')
print('=' * 60)

for idx, (topic_name, query) in enumerate(TOPICS.items(), 1):
    print(f'\n[{idx:>2}/{len(TOPICS)}] {topic_name}')
    print(f'  Query: {query}')

    articles = fetch_gdelt(topic_name, query, START_DATE, END_DATE)
    topic_summary[topic_name] = len(articles)

    if articles:
        all_articles.extend(articles)
        topic_df   = pd.DataFrame(articles)
        topic_path = os.path.join(OUTPUT_DIR, f'{topic_name.lower()}_raw.csv')
        topic_df.to_csv(topic_path, index=False)
        print(f'  ✓ {len(articles)} articles saved → {topic_path}')
    else:
        failed_topics.append(topic_name)
        print(f'  ⚠️  0 articles')

    time.sleep(SLEEP_TOPIC)

print()
print('=' * 60)
print('  FETCH COMPLETE')
print('=' * 60)
print(f'  Total articles  : {len(all_articles)}')
print(f'  Topics success  : {len(TOPICS) - len(failed_topics)} / {len(TOPICS)}')
if failed_topics:
    print(f'  Failed topics   : {failed_topics}')
print()
print('  Articles per topic:')
for t, c in sorted(topic_summary.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * min(c // 10, 25)
    print(f'  {t:<30} : {c:>4}  {bar}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Clean, Save and Download
# ─────────────────────────────────────────────────────────────────────────

if not all_articles:
    print('⚠️  No articles to save.')
else:
    master_df = pd.DataFrame(all_articles)

    # Parse dates
    master_df['published_at'] = pd.to_datetime(
        master_df['published_at'], format='%Y%m%dT%H%M%SZ', errors='coerce'
    )
    master_df['date'] = master_df['published_at'].dt.date

    # Remove duplicates
    master_df = master_df.drop_duplicates(subset=['url', 'topic'])
    master_df = master_df.sort_values('published_at', ascending=False)

    # Sentiment classification
    master_df['sentiment'] = master_df['tone'].apply(
        lambda x: 'positive' if x and x > 1 else ('negative' if x and x < -1 else 'neutral')
    )

    # Save master file
    master_path = 'step5a_raw_news_all_topics.csv'
    master_df.to_csv(master_path, index=False)

    print('=' * 60)
    print('  OUTPUT SUMMARY')
    print('=' * 60)
    print(f'  Total articles   : {len(master_df)}')
    print(f'  Unique sources   : {master_df["source"].nunique()}')
    print(f'  Date range       : {master_df["date"].min()} → {master_df["date"].max()}')
    print(f'  Columns          : {list(master_df.columns)}')
    print()
    print('  Sentiment breakdown:')
    print(master_df['sentiment'].value_counts().to_string())
    print()
    print('  Tone statistics:')
    t = master_df['tone'].dropna()
    print(f'    Mean : {t.mean():.4f}')
    print(f'    Std  : {t.std():.4f}')
    print(f'    Min  : {t.min():.4f}')
    print(f'    Max  : {t.max():.4f}')
    print()
    print('  Sample headlines (5 most recent):')
    for _, row in master_df.head(5).iterrows():
        tone_str = f"{row['tone']:.2f}" if pd.notna(row['tone']) else 'N/A'
        print(f'    [{row["topic"]:<25}] tone={tone_str:>6} | {str(row["title"])[:55]}')

    print()
    print('⬇️  Downloading master file...')
    files.download(master_path)
    print(f'  ✓ {master_path} downloaded')
    print()
    print('✅ Step 5a Complete!')
    print('  Next: tell me what you want to do in Step 5b')